# 02 - Silver: Delta para Delta

Este notebook cria dimensoes e fatos Silver a partir das tabelas Bronze em Delta.

Variaveis de ambiente esperadas:

- `AIDP_CATALOG`, `BRONZE_SCHEMA`, `SILVER_SCHEMA`
- `SILVER_WRITE_MODE`, padrao `overwrite`

### Proxima celula
Importa funcoes PySpark usadas nas transformacoes da camada Silver.


In [ ]:
import os
from pyspark.sql import functions as F

### Proxima celula
Define os parametros de catalogo, schemas Bronze/Silver e modo de escrita das tabelas Delta Silver.


In [ ]:
CATALOG = os.getenv('AIDP_CATALOG', 'main')
BRONZE_SCHEMA = os.getenv('BRONZE_SCHEMA', 'bronze')
SILVER_SCHEMA = os.getenv('SILVER_SCHEMA', 'silver')
SILVER_WRITE_MODE = os.getenv('SILVER_WRITE_MODE', 'overwrite')

### Proxima celula
Cria funcoes auxiliares para referenciar tabelas no catalogo AIDP e gravar DataFrames Silver em Delta.


In [ ]:
def q_namespace(catalog, schema):
    return f'`{catalog}`.`{schema}`'


def q_table(catalog, schema, table):
    return f'`{catalog}`.`{schema}`.`{table}`'


def bronze(table):
    return spark.table(q_table(CATALOG, BRONZE_SCHEMA, table))


def write_silver(df, table):
    (
        df.write
        .format('delta')
        .mode(SILVER_WRITE_MODE)
        .option('overwriteSchema', 'true')
        .saveAsTable(q_table(CATALOG, SILVER_SCHEMA, table))
    )
    print(f'Wrote {q_table(CATALOG, SILVER_SCHEMA, table)}')

### Proxima celula
Cria o schema Silver se necessario e carrega as tabelas Bronze que serao usadas nas transformacoes.


In [ ]:
spark.sql(f'CREATE SCHEMA IF NOT EXISTS {q_namespace(CATALOG, SILVER_SCHEMA)}')

warehouse = bronze('bronze_warehouse').alias('w')
district = bronze('bronze_district').alias('d')
customer = bronze('bronze_customer').alias('c')
history = bronze('bronze_history').alias('h')
item = bronze('bronze_item').alias('i')
stock = bronze('bronze_stock').alias('s')
orders = bronze('bronze_orders').alias('o')
new_orders = bronze('bronze_new_orders').alias('no')
order_line = bronze('bronze_order_line').alias('ol')

### Proxima celula
Cria a dimensao dim_location integrando armazens e distritos com atributos geograficos, fiscais e operacionais.


In [ ]:
dim_location = (
    district
    .join(warehouse, F.col('d.d_w_id') == F.col('w.w_id'), 'inner')
    .select(
        F.col('w.w_id').cast('int').alias('warehouse_id'),
        F.col('w.w_name').alias('warehouse_name'),
        F.col('d.d_id').cast('int').alias('district_id'),
        F.col('d.d_name').alias('district_name'),
        F.col('w.w_city').alias('warehouse_city'),
        F.col('w.w_state').alias('warehouse_state'),
        F.col('w.w_zip').alias('warehouse_zip'),
        F.col('d.d_city').alias('district_city'),
        F.col('d.d_state').alias('district_state'),
        F.col('d.d_zip').alias('district_zip'),
        F.col('w.w_tax').cast('decimal(10,4)').alias('warehouse_tax'),
        F.col('d.d_tax').cast('decimal(10,4)').alias('district_tax'),
        F.col('w.w_ytd').cast('decimal(18,2)').alias('warehouse_ytd'),
        F.col('d.d_ytd').cast('decimal(18,2)').alias('district_ytd'),
        F.col('d.d_next_o_id').cast('int').alias('next_order_id'),
    )
)

write_silver(dim_location, 'dim_location')

### Proxima celula
Cria a dimensao dim_customer com cadastro, dados de credito, saldo e indicadores historicos dos clientes.


In [ ]:
dim_customer = (
    customer
    .select(
        F.col('c_w_id').cast('int').alias('warehouse_id'),
        F.col('c_d_id').cast('int').alias('district_id'),
        F.col('c_id').cast('int').alias('customer_id'),
        F.col('c_first').alias('first_name'),
        F.col('c_middle').alias('middle_name'),
        F.col('c_last').alias('last_name'),
        F.concat_ws(' ', F.col('c_first'), F.col('c_middle'), F.col('c_last')).alias('customer_name'),
        F.col('c_street_1').alias('street_1'),
        F.col('c_street_2').alias('street_2'),
        F.col('c_city').alias('city'),
        F.col('c_state').alias('state'),
        F.col('c_zip').alias('zip'),
        F.col('c_phone').alias('phone'),
        F.col('c_since').cast('timestamp').alias('customer_since'),
        F.col('c_credit').alias('credit_status'),
        F.col('c_credit_lim').cast('decimal(18,2)').alias('credit_limit'),
        F.col('c_discount').cast('decimal(10,4)').alias('discount'),
        F.col('c_balance').cast('decimal(18,2)').alias('balance'),
        F.col('c_ytd_payment').cast('decimal(18,2)').alias('ytd_payment'),
        F.col('c_payment_cnt').cast('int').alias('payment_count'),
        F.col('c_delivery_cnt').cast('int').alias('delivery_count'),
    )
)

write_silver(dim_customer, 'dim_customer')

### Proxima celula
Cria a dimensao dim_item com o catalogo de produtos e seus precos de referencia.


In [ ]:
dim_item = (
    item
    .select(
        F.col('i_id').cast('int').alias('item_id'),
        F.col('i_im_id').cast('int').alias('item_image_id'),
        F.col('i_name').alias('item_name'),
        F.col('i_price').cast('decimal(18,2)').alias('item_price'),
        F.col('i_data').alias('item_data'),
    )
)

write_silver(dim_item, 'dim_item')

### Proxima celula
Cria o fato fact_inventory com estoque por item e armazem, incluindo valor financeiro estimado do inventario.


In [ ]:
fact_inventory = (
    stock
    .join(item, F.col('s.s_i_id') == F.col('i.i_id'), 'left')
    .select(
        F.col('s.s_w_id').cast('int').alias('warehouse_id'),
        F.col('s.s_i_id').cast('int').alias('item_id'),
        F.col('i.i_name').alias('item_name'),
        F.col('i.i_price').cast('decimal(18,2)').alias('item_price'),
        F.col('s.s_quantity').cast('int').alias('stock_quantity'),
        F.col('s.s_ytd').cast('int').alias('stock_ytd'),
        F.col('s.s_order_cnt').cast('int').alias('order_count'),
        F.col('s.s_remote_cnt').cast('int').alias('remote_count'),
        (F.col('s.s_quantity') * F.col('i.i_price')).cast('decimal(18,2)').alias('inventory_value'),
        F.col('s.s_data').alias('stock_data'),
    )
)

write_silver(fact_inventory, 'fact_inventory')

### Proxima celula
Cria o fato fact_order_header com o cabecalho dos pedidos, cliente, data, transportadora e flags operacionais.


In [ ]:
fact_order_header = (
    orders
    .join(
        customer,
        (F.col('o.o_w_id') == F.col('c.c_w_id'))
        & (F.col('o.o_d_id') == F.col('c.c_d_id'))
        & (F.col('o.o_c_id') == F.col('c.c_id')),
        'left',
    )
    .select(
        F.col('o.o_w_id').cast('int').alias('warehouse_id'),
        F.col('o.o_d_id').cast('int').alias('district_id'),
        F.col('o.o_id').cast('int').alias('order_id'),
        F.col('o.o_c_id').cast('int').alias('customer_id'),
        F.concat_ws(' ', F.col('c.c_first'), F.col('c.c_middle'), F.col('c.c_last')).alias('customer_name'),
        F.col('o.o_entry_d').cast('timestamp').alias('order_entry_timestamp'),
        F.to_date(F.col('o.o_entry_d')).alias('order_date'),
        F.col('o.o_carrier_id').cast('int').alias('carrier_id'),
        F.col('o.o_ol_cnt').cast('int').alias('order_line_count'),
        F.col('o.o_all_local').cast('int').alias('all_local_flag'),
    )
)

write_silver(fact_order_header, 'fact_order_header')

### Proxima celula
Cria o fato fact_order_line com itens de pedido, quantidades, valores, preco unitario e informacoes de entrega.


In [ ]:
fact_order_line = (
    order_line
    .join(
        orders,
        (F.col('ol.ol_w_id') == F.col('o.o_w_id'))
        & (F.col('ol.ol_d_id') == F.col('o.o_d_id'))
        & (F.col('ol.ol_o_id') == F.col('o.o_id')),
        'left',
    )
    .join(item, F.col('ol.ol_i_id') == F.col('i.i_id'), 'left')
    .select(
        F.col('ol.ol_w_id').cast('int').alias('warehouse_id'),
        F.col('ol.ol_d_id').cast('int').alias('district_id'),
        F.col('ol.ol_o_id').cast('int').alias('order_id'),
        F.col('o.o_c_id').cast('int').alias('customer_id'),
        F.col('ol.ol_number').cast('int').alias('order_line_number'),
        F.col('ol.ol_i_id').cast('int').alias('item_id'),
        F.col('i.i_name').alias('item_name'),
        F.col('i.i_price').cast('decimal(18,2)').alias('catalog_unit_price'),
        F.col('ol.ol_supply_w_id').cast('int').alias('supply_warehouse_id'),
        F.col('o.o_entry_d').cast('timestamp').alias('order_entry_timestamp'),
        F.to_date(F.col('o.o_entry_d')).alias('order_date'),
        F.col('ol.ol_delivery_d').cast('timestamp').alias('delivery_timestamp'),
        F.col('ol.ol_quantity').cast('int').alias('quantity'),
        F.col('ol.ol_amount').cast('decimal(18,2)').alias('amount'),
        F.when(
            F.col('ol.ol_quantity') != 0,
            F.col('ol.ol_amount') / F.col('ol.ol_quantity'),
        ).cast('decimal(18,2)').alias('unit_price'),
        (F.col('ol.ol_quantity') * F.col('i.i_price')).cast('decimal(18,2)').alias('gross_amount'),
        F.col('ol.ol_dist_info').alias('distribution_info'),
    )
)

write_silver(fact_order_line, 'fact_order_line')

### Proxima celula
Cria o fato fact_payment com pagamentos de clientes, datas, valores e relacao com localidade operacional.


In [ ]:
fact_payment = (
    history
    .join(
        customer,
        (F.col('h.h_c_w_id') == F.col('c.c_w_id'))
        & (F.col('h.h_c_d_id') == F.col('c.c_d_id'))
        & (F.col('h.h_c_id') == F.col('c.c_id')),
        'left',
    )
    .select(
        F.col('h.h_id').cast('long').alias('payment_id'),
        F.col('h.h_c_w_id').cast('int').alias('customer_warehouse_id'),
        F.col('h.h_c_d_id').cast('int').alias('customer_district_id'),
        F.col('h.h_c_id').cast('int').alias('customer_id'),
        F.col('h.h_w_id').cast('int').alias('warehouse_id'),
        F.col('h.h_d_id').cast('int').alias('district_id'),
        F.concat_ws(' ', F.col('c.c_first'), F.col('c.c_middle'), F.col('c.c_last')).alias('customer_name'),
        F.col('h.h_date').cast('timestamp').alias('payment_timestamp'),
        F.to_date(F.col('h.h_date')).alias('payment_date'),
        F.col('h.h_amount').cast('decimal(18,2)').alias('payment_amount'),
        F.col('h.h_data').alias('payment_data'),
    )
)

write_silver(fact_payment, 'fact_payment')

### Proxima celula
Cria o fato fact_pending_order com pedidos pendentes e idade do backlog calculada em dias.


In [ ]:
fact_pending_order = (
    new_orders
    .join(
        orders,
        (F.col('no.no_w_id') == F.col('o.o_w_id'))
        & (F.col('no.no_d_id') == F.col('o.o_d_id'))
        & (F.col('no.no_o_id') == F.col('o.o_id')),
        'left',
    )
    .select(
        F.col('no.no_w_id').cast('int').alias('warehouse_id'),
        F.col('no.no_d_id').cast('int').alias('district_id'),
        F.col('no.no_o_id').cast('int').alias('order_id'),
        F.col('o.o_c_id').cast('int').alias('customer_id'),
        F.col('o.o_entry_d').cast('timestamp').alias('order_entry_timestamp'),
        F.to_date(F.col('o.o_entry_d')).alias('order_date'),
        F.datediff(F.current_date(), F.to_date(F.col('o.o_entry_d'))).alias('pending_age_days'),
        F.lit('PENDING').alias('pending_status'),
    )
)

write_silver(fact_pending_order, 'fact_pending_order')

### Proxima celula
Monta um payload JSON de sucesso para uso em workflows AIDP ou para exibicao quando executado interativamente.


In [ ]:
import json

payload = {
    'status': 'SUCCESS',
    'layer': 'silver',
    'catalog': CATALOG,
    'schema': SILVER_SCHEMA,
    'tables': [
        'dim_location',
        'dim_customer',
        'dim_item',
        'fact_inventory',
        'fact_order_header',
        'fact_order_line',
        'fact_payment',
        'fact_pending_order',
    ],
}

try:
    oidlUtils.notebook.exit(json.dumps(payload))
except NameError:
    print(json.dumps(payload))